# Decoder Reliability Analysis From Next-Token Traces

This notebook is for reliability work on decoder next-token traces. The goal is not only to ask whether the final top-1 token is correct, but to identify trace-derived statistics that separate:

- `top1_correct`
- `near_miss_topk`: top-1 is wrong, but the true token is still inside the model's final top-k set
- `hard_miss`: the true token is absent from the model's final top-k set

The notebook is organized around two distinct use cases:

1. `Observable reliability features`: statistics available at inference time without knowing the true token. These are the features that can actually be used for error prediction or post-hoc validation in deployment.
2. `Truth-aware diagnostics`: statistics that use the true token only to understand what kind of error happened. These are useful for analysis, but they are not valid deployment features.

The layerwise trace analysis uses a candidate-set logit lens. Instead of scoring the full vocabulary at every layer, it scores a small decision set:

- for observable features: the model's final top-k tokens
- for truth-aware diagnostics: the final top-k tokens plus the true next token

This keeps the analysis tractable while still capturing the local decision geometry that matters for reliability.

In [ ]:
from pathlib import Path
import ast
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

from src.models.load import load_base
from src.traces_utils.store import TraceStore

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

## Configuration

`RUN_DIR` should point at a decoder trace directory produced by `extract_trace_nexttok_dec.py` with at least:

- `tokens.parquet`
- `dec_hidden.zarr`
- `dec_res_pre_attn.zarr`
- `dec_res_post_attn.zarr`
- `dec_res_post_mlp.zarr`

`MODEL_SOURCE` should be the local fine-tuned checkpoint directory or a Hugging Face model ID that matches the trace run. If it is left as `None`, the notebook will try to use `meta.json["model"]`.

In [ ]:
RUN_DIR = Path(r"C:\traces\wt2_val_full_ft_medium_hidden_resid")
MODEL_SOURCE = None

TRACE_MAX_EXAMPLES = None
NEAR_K = 5
CANDIDATE_TOPK = 5
EARLY_LAYER_FRACTION = 0.5
LAST_N = 5
CV_SPLITS = 5
SEED = 42

assert RUN_DIR.exists(), f"Trace run not found: {RUN_DIR}"

## Data Read-In

This section reads the trace store, checks that the arrays needed for the analysis exist, and constructs the full-dataset output-space table that later sections build on.

In [ ]:
st = TraceStore(str(RUN_DIR))
arr = st.arrays()
df = st.tokens.copy().reset_index(drop=True)
valid_df = df[df["next_pos"] >= 0].copy().reset_index(drop=True)

print("meta:")
print(json.dumps(st.meta, indent=2))
print()
print("arrays:", arr)
print("rows:", len(df))
print("valid next-token rows:", len(valid_df))

assert "dec_hidden" in arr, "Trace has no decoder hidden states."
assert "dec_res_pre_attn" in arr, "Trace has no decoder pre-attention residual stream."
assert "dec_res_post_attn" in arr, "Trace has no decoder post-attention residual stream."
assert "dec_res_post_mlp" in arr, "Trace has no decoder post-MLP residual stream."

In [ ]:
def listify(x):
    if isinstance(x, list):
        return x
    if isinstance(x, tuple):
        return list(x)
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, str):
        s = x.strip()
        if s.startswith("[") and s.endswith("]"):
            return list(ast.literal_eval(s))
    if pd.isna(x):
        return []
    return [x]


out_df = valid_df.copy()
out_df["topk_ids"] = out_df["next_topk_ids"].map(listify)
out_df["topk_probs"] = out_df["next_topk_probs"].map(listify)

out_df["top1_correct"] = out_df["next_correct"].astype(bool)
out_df["top1_prob"] = out_df["next_pred_prob"].astype(float)
out_df["true_prob"] = out_df["next_true_prob"].astype(float)
out_df["entropy"] = out_df["next_entropy"].astype(float)

def top1_top2_prob_gap(probs):
    probs = list(probs)
    if len(probs) < 2:
        return np.nan
    return float(probs[0] - probs[1])


def top1_top2_logit_gap(probs):
    probs = list(probs)
    if len(probs) < 2:
        return np.nan
    p1 = max(float(probs[0]), 1e-12)
    p2 = max(float(probs[1]), 1e-12)
    return float(np.log(p1) - np.log(p2))


def topk_mass(probs, k):
    probs = list(probs)
    return float(np.sum(probs[:k]))


def topk_entropy_norm(probs):
    probs = np.asarray(list(probs), dtype=float)
    if len(probs) <= 1:
        return 0.0
    probs = np.clip(probs, 1e-12, None)
    ent = float(-(probs * np.log(probs)).sum())
    return ent / float(np.log(len(probs)))


def true_in_topk(row, k):
    return int(row["next_true_id"]) in [int(x) for x in row["topk_ids"][:k]]


def true_rank_topk(row, k):
    ids = [int(x) for x in row["topk_ids"][:k]]
    true_id = int(row["next_true_id"])
    return ids.index(true_id) + 1 if true_id in ids else k + 1


out_df["top1_top2_prob_gap"] = out_df["topk_probs"].map(top1_top2_prob_gap)
out_df["top1_top2_logit_gap"] = out_df["topk_probs"].map(top1_top2_logit_gap)
out_df["topk_mass"] = out_df["topk_probs"].map(lambda p: topk_mass(p, NEAR_K))
out_df["tail_mass"] = 1.0 - out_df["topk_mass"]
out_df["topk_entropy_norm"] = out_df["topk_probs"].map(topk_entropy_norm)
out_df["true_in_topk"] = out_df.apply(lambda r: true_in_topk(r, NEAR_K), axis=1)
out_df["true_rank_topk"] = out_df.apply(lambda r: true_rank_topk(r, NEAR_K), axis=1)
out_df["pred_surprisal"] = -np.log(np.clip(out_df["top1_prob"], 1e-12, None))
out_df["true_surprisal"] = -np.log(np.clip(out_df["true_prob"], 1e-12, None))
out_df["true_prob_ratio"] = out_df["true_prob"] / np.clip(out_df["top1_prob"], 1e-12, None)

def error_bucket(row):
    if bool(row["top1_correct"]):
        return "top1_correct"
    if bool(row["true_in_topk"]):
        return "near_miss_topk"
    return "hard_miss"


bucket_order = ["top1_correct", "near_miss_topk", "hard_miss"]
bucket_colors = {
    "top1_correct": "#2a9d8f",
    "near_miss_topk": "#e9c46a",
    "hard_miss": "#e76f51",
}

out_df["error_bucket"] = out_df.apply(error_bucket, axis=1)
out_df["is_top1_error"] = ~out_df["top1_correct"]
out_df["is_hard_miss"] = out_df["error_bucket"].eq("hard_miss")

display(
    pd.DataFrame(
        {
            "count": out_df["error_bucket"].value_counts().reindex(bucket_order),
            "share": out_df["error_bucket"].value_counts(normalize=True).reindex(bucket_order),
        }
    ).round(4)
)

print("top-1 accuracy:", round(float(out_df["top1_correct"].mean()), 4))
print(f"hit@{NEAR_K}:", round(float(out_df["true_in_topk"].mean()), 4))

In [ ]:
model_source = MODEL_SOURCE or st.meta.get("model", "gpt2")
source_path = Path(str(model_source))

if source_path.exists():
    tok, model, device = load_base(model_path=str(source_path))
else:
    tok, model, device = load_base(str(model_source))

model.eval()
assert hasattr(model, "lm_head"), "Expected decoder LM with lm_head."
assert hasattr(model, "transformer") and hasattr(model.transformer, "ln_f"), "Expected GPT-style final layer norm."

W = model.lm_head.weight.detach().cpu()
ln_f = model.transformer.ln_f
ln_device = ln_f.weight.device

L_plus_1 = int(arr["dec_hidden"][1])
L = L_plus_1 - 1
HALF_HIDDEN = max(2, int(np.ceil(EARLY_LAYER_FRACTION * L_plus_1)))
HALF_RESID = max(1, int(np.ceil(EARLY_LAYER_FRACTION * L)))

print("model source:", model_source)
print("device:", device)
print("L+1:", L_plus_1)
print("L:", L)
print("half-depth hidden cutoff:", HALF_HIDDEN)
print("half-depth residual cutoff:", HALF_RESID)

## Section 1. Full-Dataset Output-Space Reliability

These features come directly from `tokens.parquet`. They are cheap, full-dataset, and already useful for post-prediction validation. They do **not** use hidden states or residual streams.

In [ ]:
output_feature_cols = [
    "top1_prob",
    "entropy",
    "top1_top2_prob_gap",
    "top1_top2_logit_gap",
    "topk_mass",
    "tail_mass",
    "topk_entropy_norm",
    "pred_surprisal",
    "true_prob",
    "true_prob_ratio",
    "true_rank_topk",
]

bucket_summary = (
    out_df.groupby("error_bucket")[output_feature_cols]
    .agg(["mean", "median", "std"])
    .round(4)
)

display(bucket_summary)

In [ ]:
plot_cols = [
    "top1_prob",
    "top1_top2_logit_gap",
    "entropy",
    "topk_mass",
    "topk_entropy_norm",
    "pred_surprisal",
]

plot_titles = {
    "top1_prob": "Final top-1 probability",
    "top1_top2_logit_gap": "Final top1-top2 logit gap",
    "entropy": "Final entropy",
    "topk_mass": f"Top-{NEAR_K} probability mass",
    "topk_entropy_norm": "Normalized entropy inside stored top-k",
    "pred_surprisal": "Predicted-token surprisal",
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, col in zip(axes, plot_cols):
    for bucket in bucket_order:
        vals = pd.to_numeric(
            out_df.loc[out_df["error_bucket"].eq(bucket), col],
            errors="coerce",
        ).dropna()
        ax.hist(
            vals,
            bins=45,
            density=True,
            alpha=0.35,
            color=bucket_colors[bucket],
            label=bucket,
        )
    ax.set_title(plot_titles[col])
    ax.set_xlabel(col)

axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
def calibration_table(prob, is_correct, n_bins=12):
    prob = np.asarray(prob, dtype=float)
    is_correct = np.asarray(is_correct, dtype=bool)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    for i in range(n_bins):
        lo = bins[i]
        hi = bins[i + 1]
        if i == n_bins - 1:
            mask = (prob >= lo) & (prob <= hi)
        else:
            mask = (prob >= lo) & (prob < hi)
        if not mask.any():
            continue
        rows.append(
            {
                "bin_left": lo,
                "bin_right": hi,
                "count": int(mask.sum()),
                "mean_conf": float(prob[mask].mean()),
                "emp_acc": float(is_correct[mask].mean()),
                "emp_error": float((~is_correct[mask]).mean()),
            }
        )
    return pd.DataFrame(rows)


def selective_accuracy(confidence, is_correct, n_points=30):
    confidence = np.asarray(confidence, dtype=float)
    is_correct = np.asarray(is_correct, dtype=bool)
    order = np.argsort(-confidence)
    conf_sorted = confidence[order]
    ok_sorted = is_correct[order]
    rows = []
    for cov in np.linspace(0.05, 1.0, n_points):
        keep = max(1, int(round(cov * len(conf_sorted))))
        rows.append(
            {
                "coverage": cov,
                "accuracy": float(ok_sorted[:keep].mean()),
                "error": float((~ok_sorted[:keep]).mean()),
            }
        )
    return pd.DataFrame(rows)


cal_df = calibration_table(out_df["top1_prob"], out_df["top1_correct"], n_bins=12)

sel_top1_prob = selective_accuracy(out_df["top1_prob"], out_df["top1_correct"])
sel_margin = selective_accuracy(out_df["top1_top2_logit_gap"], out_df["top1_correct"])
sel_low_entropy = selective_accuracy(-out_df["entropy"], out_df["top1_correct"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot([0, 1], [0, 1], color="black", linewidth=1, linestyle="--")
axes[0].plot(cal_df["mean_conf"], cal_df["emp_acc"], marker="o")
axes[0].set_title("Reliability diagram for final top-1 probability")
axes[0].set_xlabel("Mean predicted confidence")
axes[0].set_ylabel("Empirical accuracy")

axes[1].plot(sel_top1_prob["coverage"], sel_top1_prob["accuracy"], label="top1_prob")
axes[1].plot(sel_margin["coverage"], sel_margin["accuracy"], label="top1_top2_logit_gap")
axes[1].plot(sel_low_entropy["coverage"], sel_low_entropy["accuracy"], label="-entropy")
axes[1].set_title("Selective prediction: accuracy vs coverage")
axes[1].set_xlabel("Coverage kept")
axes[1].set_ylabel("Accuracy among kept examples")
axes[1].legend()

plt.tight_layout()
plt.show()

display(cal_df.round(4))

## Section 2. Observable Trace Features

This section extracts features from the hidden and residual traces that are **valid deployment features**. They only use information available from the model's own trace:

- the final predicted top-k tokens
- layerwise competition among those tokens
- stability of the eventual top-1 token across layers
- residual-stage gains for the eventual top-1 margin and entropy

No true-token information is used in these features.

In [ ]:
def softmax_rows(logits):
    logits = np.asarray(logits, dtype=np.float32)
    logits = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(logits)
    return probs / probs.sum(axis=1, keepdims=True)


def normalized_entropy_rows(probs):
    probs = np.asarray(probs, dtype=np.float32)
    if probs.shape[1] <= 1:
        return np.zeros(probs.shape[0], dtype=np.float32)
    probs = np.clip(probs, 1e-12, None)
    ent = -(probs * np.log(probs)).sum(axis=1)
    return ent / float(np.log(probs.shape[1]))


def slope(arr):
    arr = np.asarray(arr, dtype=np.float32)
    if len(arr) < 2:
        return np.nan
    x = np.arange(len(arr), dtype=np.float32)
    x = x - x.mean()
    denom = float((x * x).sum())
    if denom <= 0:
        return np.nan
    return float((x * arr).sum() / denom)


def slope_last(arr, n=LAST_N):
    arr = np.asarray(arr, dtype=np.float32)
    n = min(len(arr), n)
    return slope(arr[-n:])


def first_layer(mask):
    idx = np.flatnonzero(np.asarray(mask, dtype=bool))
    return float(idx[0]) if len(idx) else np.nan


def first_stable_layer(mask):
    mask = np.asarray(mask, dtype=bool)
    for i in range(len(mask)):
        if mask[i] and mask[i:].all():
            return float(i)
    return np.nan


def project_hidden_stack(hidden_stack):
    hidden_stack = np.asarray(hidden_stack, dtype=np.float32)
    if len(hidden_stack) == 1:
        return hidden_stack.copy()
    pre = torch.as_tensor(hidden_stack[:-1], dtype=torch.float32, device=ln_device)
    pre = ln_f(pre).detach().cpu().numpy()
    final = hidden_stack[-1:].astype(np.float32, copy=False)
    return np.concatenate([pre, final], axis=0)


def project_resid_stack(resid_stack):
    resid_stack = np.asarray(resid_stack, dtype=np.float32)
    t = torch.as_tensor(resid_stack, dtype=torch.float32, device=ln_device)
    return ln_f(t).detach().cpu().numpy()


def select_weights(token_ids):
    idx = torch.as_tensor(token_ids, dtype=torch.long)
    return W[idx].numpy()


def candidate_topk_ids(row, k=CANDIDATE_TOPK):
    ids = []
    for tid in row["topk_ids"][:k]:
        tid = int(tid)
        if tid not in ids:
            ids.append(tid)
    if not ids:
        ids = [int(row["next_pred_id"])]
    return ids


def candidate_with_true_ids(row, k=CANDIDATE_TOPK):
    ids = [int(row["next_true_id"])]
    for tid in candidate_topk_ids(row, k=k):
        if tid not in ids:
            ids.append(tid)
    return ids


def observable_curves(logits):
    probs = softmax_rows(logits)
    entropy = normalized_entropy_rows(probs)
    winner = logits.argmax(axis=1)

    if logits.shape[1] >= 2:
        margin = logits[:, 0] - np.max(logits[:, 1:], axis=1)
        order = np.argsort(logits, axis=1)
        runnerup = order[:, -2]
    else:
        margin = logits[:, 0].copy()
        runnerup = np.full(logits.shape[0], -1, dtype=int)

    top1_prob = probs[:, 0]
    return top1_prob, entropy, margin, winner, runnerup


def truth_curves(logits, anchor_idx=0):
    probs = softmax_rows(logits)
    rank = 1 + (logits > logits[:, [anchor_idx]]).sum(axis=1)
    winner = logits.argmax(axis=1)
    if logits.shape[1] >= 2:
        best_other = logits.copy()
        best_other[:, anchor_idx] = -np.inf
        margin = logits[:, anchor_idx] - best_other.max(axis=1)
    else:
        margin = logits[:, anchor_idx].copy()
    return probs[:, anchor_idx], rank.astype(np.float32), margin.astype(np.float32), winner

In [ ]:
def extract_trace_features(row):
    eid = row["example_id"]
    pos = int(row["next_pos"])

    H = st.hidden(eid, side="dec")[:, pos, :]
    H_proj = project_hidden_stack(H)

    pred_ids = candidate_topk_ids(row, k=CANDIDATE_TOPK)
    pred_W = select_weights(pred_ids)
    pred_logits = H_proj @ pred_W.T

    obs_top1_prob, obs_entropy, obs_margin, obs_winner, obs_runnerup = observable_curves(pred_logits)
    obs_final_top1_agree = (obs_winner == 0)

    diag_ids = candidate_with_true_ids(row, k=CANDIDATE_TOPK)
    diag_W = select_weights(diag_ids)
    diag_logits = H_proj @ diag_W.T
    diag_true_prob, diag_true_rank, diag_true_margin, diag_winner = truth_curves(diag_logits, anchor_idx=0)

    R_pre = project_resid_stack(st.resid(eid, stage="pre_attn", side="dec")[:, pos, :])
    R_attn = project_resid_stack(st.resid(eid, stage="post_attn", side="dec")[:, pos, :])
    R_post = project_resid_stack(st.resid(eid, stage="post_mlp", side="dec")[:, pos, :])

    pre_logits = R_pre @ pred_W.T
    attn_logits = R_attn @ pred_W.T
    post_logits = R_post @ pred_W.T

    pre_top1_prob, pre_entropy, pre_margin, pre_winner, _ = observable_curves(pre_logits)
    attn_top1_prob, attn_entropy, attn_margin, attn_winner, _ = observable_curves(attn_logits)
    post_top1_prob, post_entropy, post_margin, post_winner, _ = observable_curves(post_logits)

    attn_margin_gain = attn_margin - pre_margin
    mlp_margin_gain = post_margin - attn_margin
    attn_top1_prob_gain = attn_top1_prob - pre_top1_prob
    mlp_top1_prob_gain = post_top1_prob - attn_top1_prob
    attn_entropy_drop = pre_entropy - attn_entropy
    mlp_entropy_drop = attn_entropy - post_entropy

    return {
        "example_id": eid,
        "obs_margin_curve": obs_margin,
        "obs_entropy_curve": obs_entropy,
        "obs_top1_prob_curve": obs_top1_prob,
        "obs_final_top1_agree_curve": obs_final_top1_agree.astype(np.float32),
        "obs_attn_margin_gain_curve": attn_margin_gain,
        "obs_mlp_margin_gain_curve": mlp_margin_gain,
        "obs_attn_entropy_drop_curve": attn_entropy_drop,
        "obs_mlp_entropy_drop_curve": mlp_entropy_drop,
        "diag_true_prob_curve": diag_true_prob,
        "diag_true_rank_curve": diag_true_rank,
        "diag_true_margin_curve": diag_true_margin,
        "obs_margin_early_mean": float(obs_margin[:HALF_HIDDEN].mean()),
        "obs_margin_early_slope": slope(obs_margin[:HALF_HIDDEN]),
        "obs_margin_final": float(obs_margin[-1]),
        "obs_margin_last5_mean": float(obs_margin[-min(LAST_N, len(obs_margin)):].mean()),
        "obs_margin_last5_slope": slope_last(obs_margin, LAST_N),
        "obs_entropy_early_mean": float(obs_entropy[:HALF_HIDDEN].mean()),
        "obs_entropy_early_slope": slope(obs_entropy[:HALF_HIDDEN]),
        "obs_entropy_final": float(obs_entropy[-1]),
        "obs_entropy_last5_mean": float(obs_entropy[-min(LAST_N, len(obs_entropy)):].mean()),
        "obs_entropy_last5_slope": slope_last(obs_entropy, LAST_N),
        "obs_top1_prob_early_mean": float(obs_top1_prob[:HALF_HIDDEN].mean()),
        "obs_top1_prob_early_slope": slope(obs_top1_prob[:HALF_HIDDEN]),
        "obs_top1_prob_final": float(obs_top1_prob[-1]),
        "obs_top1_prob_last5_slope": slope_last(obs_top1_prob, LAST_N),
        "obs_winner_flip_count": int(np.sum(obs_winner[1:] != obs_winner[:-1])),
        "obs_winner_flip_early": int(np.sum(obs_winner[1:HALF_HIDDEN] != obs_winner[:HALF_HIDDEN - 1])),
        "obs_runnerup_switch_count": int(np.sum(obs_runnerup[1:] != obs_runnerup[:-1])),
        "obs_final_top1_first_layer": first_layer(obs_final_top1_agree),
        "obs_final_top1_stable_layer": first_stable_layer(obs_final_top1_agree),
        "obs_final_top1_agreement_frac": float(obs_final_top1_agree.mean()),
        "obs_final_top1_agreement_early": float(obs_final_top1_agree[:HALF_HIDDEN].mean()),
        "obs_attn_margin_gain_early_mean": float(attn_margin_gain[:HALF_RESID].mean()),
        "obs_attn_margin_gain_mean": float(attn_margin_gain.mean()),
        "obs_attn_margin_gain_last5_mean": float(attn_margin_gain[-min(LAST_N, len(attn_margin_gain)):].mean()),
        "obs_mlp_margin_gain_early_mean": float(mlp_margin_gain[:HALF_RESID].mean()),
        "obs_mlp_margin_gain_mean": float(mlp_margin_gain.mean()),
        "obs_mlp_margin_gain_last5_mean": float(mlp_margin_gain[-min(LAST_N, len(mlp_margin_gain)):].mean()),
        "obs_attn_top1_prob_gain_mean": float(attn_top1_prob_gain.mean()),
        "obs_mlp_top1_prob_gain_mean": float(mlp_top1_prob_gain.mean()),
        "obs_attn_entropy_drop_early_mean": float(attn_entropy_drop[:HALF_RESID].mean()),
        "obs_attn_entropy_drop_mean": float(attn_entropy_drop.mean()),
        "obs_mlp_entropy_drop_early_mean": float(mlp_entropy_drop[:HALF_RESID].mean()),
        "obs_mlp_entropy_drop_mean": float(mlp_entropy_drop.mean()),
        "obs_attn_negative_steps": int(np.sum(attn_margin_gain < 0)),
        "obs_mlp_negative_steps": int(np.sum(mlp_margin_gain < 0)),
        "diag_true_prob_final": float(diag_true_prob[-1]),
        "diag_true_prob_peak": float(diag_true_prob.max()),
        "diag_true_rank_final": float(diag_true_rank[-1]),
        "diag_true_rank_best": float(diag_true_rank.min()),
        "diag_true_margin_final": float(diag_true_margin[-1]),
        "diag_true_margin_peak": float(diag_true_margin.max()),
        "diag_true_first_winner_layer": first_layer(diag_winner == 0),
    }


trace_source_df = out_df.copy()
if TRACE_MAX_EXAMPLES is not None:
    trace_source_df = trace_source_df.head(int(TRACE_MAX_EXAMPLES)).reset_index(drop=True)

trace_rows = []
for _, row in tqdm(trace_source_df.iterrows(), total=len(trace_source_df), desc="trace features"):
    trace_rows.append(extract_trace_features(row))

trace_feat_df = pd.DataFrame(trace_rows)
analysis_df = trace_source_df.merge(trace_feat_df, on="example_id", how="inner")

print("trace subset size:", len(analysis_df))
display(
    pd.DataFrame(
        {
            "count": analysis_df["error_bucket"].value_counts().reindex(bucket_order),
            "share": analysis_df["error_bucket"].value_counts(normalize=True).reindex(bucket_order),
        }
    ).round(4)
)

In [ ]:
observable_output_cols = [
    "top1_prob",
    "entropy",
    "top1_top2_logit_gap",
    "topk_mass",
    "tail_mass",
    "topk_entropy_norm",
    "pred_surprisal",
]

observable_trace_cols = [
    "obs_margin_early_mean",
    "obs_margin_early_slope",
    "obs_margin_final",
    "obs_margin_last5_mean",
    "obs_margin_last5_slope",
    "obs_entropy_early_mean",
    "obs_entropy_early_slope",
    "obs_entropy_final",
    "obs_entropy_last5_mean",
    "obs_entropy_last5_slope",
    "obs_top1_prob_early_mean",
    "obs_top1_prob_early_slope",
    "obs_top1_prob_final",
    "obs_top1_prob_last5_slope",
    "obs_winner_flip_count",
    "obs_winner_flip_early",
    "obs_runnerup_switch_count",
    "obs_final_top1_first_layer",
    "obs_final_top1_stable_layer",
    "obs_final_top1_agreement_frac",
    "obs_final_top1_agreement_early",
]

observable_resid_cols = [
    "obs_attn_margin_gain_early_mean",
    "obs_attn_margin_gain_mean",
    "obs_attn_margin_gain_last5_mean",
    "obs_mlp_margin_gain_early_mean",
    "obs_mlp_margin_gain_mean",
    "obs_mlp_margin_gain_last5_mean",
    "obs_attn_top1_prob_gain_mean",
    "obs_mlp_top1_prob_gain_mean",
    "obs_attn_entropy_drop_early_mean",
    "obs_attn_entropy_drop_mean",
    "obs_mlp_entropy_drop_early_mean",
    "obs_mlp_entropy_drop_mean",
    "obs_attn_negative_steps",
    "obs_mlp_negative_steps",
]

observable_all_cols = observable_output_cols + observable_trace_cols + observable_resid_cols


def single_feature_auc(frame, cols, target):
    rows = []
    y = frame[target].astype(int).to_numpy()

    family_map = {}
    for c in observable_output_cols:
        family_map[c] = "output"
    for c in observable_trace_cols:
        family_map[c] = "hidden_trace"
    for c in observable_resid_cols:
        family_map[c] = "residual_trace"

    for col in cols:
        x = pd.to_numeric(frame[col], errors="coerce").to_numpy(dtype=float)
        mask = np.isfinite(x)
        if mask.sum() < 30:
            continue
        y_m = y[mask]
        x_m = x[mask]
        if len(np.unique(y_m)) < 2:
            continue
        auc_hi = roc_auc_score(y_m, x_m)
        auc_lo = roc_auc_score(y_m, -x_m)
        if auc_lo > auc_hi:
            auc = auc_lo
            direction = "lower => more error"
            score = -x_m
        else:
            auc = auc_hi
            direction = "higher => more error"
            score = x_m
        ap = average_precision_score(y_m, score)
        rows.append(
            {
                "feature": col,
                "family": family_map[col],
                "single_feature_auroc": auc,
                "single_feature_ap": ap,
                "direction": direction,
            }
        )

    return pd.DataFrame(rows).sort_values(["single_feature_auroc", "single_feature_ap"], ascending=False)


rank_error = single_feature_auc(analysis_df, observable_all_cols, "is_top1_error")
rank_hard = single_feature_auc(analysis_df, observable_all_cols, "is_hard_miss")

print("Top single-feature separators for top-1 error")
display(rank_error.head(20).round(4))

print("Top single-feature separators for hard miss")
display(rank_hard.head(20).round(4))

In [ ]:
selected_obs_cols = [
    "top1_prob",
    "top1_top2_logit_gap",
    "obs_margin_final",
    "obs_final_top1_stable_layer",
    "obs_winner_flip_count",
    "obs_attn_margin_gain_mean",
]

selected_titles = {
    "top1_prob": "Final top-1 probability",
    "top1_top2_logit_gap": "Final top1-top2 logit gap",
    "obs_margin_final": "Final fixed-topk top1 margin",
    "obs_final_top1_stable_layer": "Layer where final top1 becomes stable",
    "obs_winner_flip_count": "Winner flips across layers",
    "obs_attn_margin_gain_mean": "Mean attention-stage margin gain",
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, col in zip(axes, selected_obs_cols):
    for bucket in bucket_order:
        vals = pd.to_numeric(
            analysis_df.loc[analysis_df["error_bucket"].eq(bucket), col],
            errors="coerce",
        ).dropna()
        ax.hist(
            vals,
            bins=40,
            density=True,
            alpha=0.35,
            label=bucket,
            color=bucket_colors[bucket],
        )
    ax.set_title(selected_titles[col])
    ax.set_xlabel(col)

axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
layers = np.arange(L_plus_1)
res_layers = np.arange(L)


def stack_curve(frame, col):
    return np.stack(frame[col].to_list(), axis=0)


def bucket_mean_curve(frame, col, bucket):
    mat = stack_curve(frame, col)
    mask = frame["error_bucket"].eq(bucket).to_numpy()
    return mat[mask].mean(axis=0)


curve_specs = [
    ("obs_margin_curve", layers, "Fixed-topk top1 margin over layers"),
    ("obs_entropy_curve", layers, "Fixed-topk entropy over layers"),
    ("obs_top1_prob_curve", layers, "Fixed-topk top1 probability over layers"),
    ("obs_final_top1_agree_curve", layers, "Fraction already aligned with final top1"),
    ("obs_attn_margin_gain_curve", res_layers, "Attention-stage gain in top1 margin"),
    ("obs_mlp_margin_gain_curve", res_layers, "MLP-stage gain in top1 margin"),
]

fig, axes = plt.subplots(3, 2, figsize=(16, 13))
axes = axes.ravel()

for ax, (col, xs, title) in zip(axes, curve_specs):
    for bucket in bucket_order:
        ax.plot(
            xs,
            bucket_mean_curve(analysis_df, col, bucket),
            label=bucket,
            color=bucket_colors[bucket],
        )
    ax.set_title(title)
    ax.set_xlabel("Layer")

axes[0].legend()
plt.tight_layout()
plt.show()

## Section 3. Truth-Aware Diagnostics

This section uses the true next token only to interpret the structure of errors. These are **not** valid deployment features, but they are valuable for understanding whether an error was a near miss or a genuine failure of the decision trace.

In [ ]:
diag_cols = [
    "diag_true_prob_final",
    "diag_true_prob_peak",
    "diag_true_rank_final",
    "diag_true_rank_best",
    "diag_true_margin_final",
    "diag_true_margin_peak",
    "diag_true_first_winner_layer",
    "true_rank_topk",
]

diag_summary = (
    analysis_df.groupby("error_bucket")[diag_cols]
    .agg(["mean", "median", "std"])
    .round(4)
)

display(diag_summary)

In [ ]:
diag_curve_specs = [
    ("diag_true_prob_curve", layers, "True-token probability over layers"),
    ("diag_true_rank_curve", layers, "True-token rank over layers (lower is better)"),
    ("diag_true_margin_curve", layers, "True-vs-best-other margin over layers"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (col, xs, title) in zip(axes, diag_curve_specs):
    for bucket in bucket_order:
        ax.plot(
            xs,
            bucket_mean_curve(analysis_df, col, bucket),
            label=bucket,
            color=bucket_colors[bucket],
        )
    ax.set_title(title)
    ax.set_xlabel("Layer")

axes[0].legend()
plt.tight_layout()
plt.show()

## Section 4. Predictive Models

This section turns the observable features into actual reliability models.

- `output_only` uses only final output-space summaries.
- `early_trace_only` uses only early/mid-layer observable trace features.
- `output_plus_early_trace` asks whether early traces add value beyond final softmax confidence.
- `output_plus_full_trace` is the strongest post-hoc validator in this notebook.

The targets are:

- `is_top1_error`
- `is_hard_miss`

In [ ]:
early_trace_features = [
    "obs_margin_early_mean",
    "obs_margin_early_slope",
    "obs_entropy_early_mean",
    "obs_entropy_early_slope",
    "obs_top1_prob_early_mean",
    "obs_top1_prob_early_slope",
    "obs_winner_flip_early",
    "obs_final_top1_agreement_early",
    "obs_attn_margin_gain_early_mean",
    "obs_mlp_margin_gain_early_mean",
    "obs_attn_entropy_drop_early_mean",
    "obs_mlp_entropy_drop_early_mean",
]

full_trace_features = observable_trace_cols + observable_resid_cols


def build_pipeline():
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "clf",
                LogisticRegression(
                    max_iter=5000,
                    class_weight="balanced",
                    random_state=SEED,
                ),
            ),
        ]
    )


def cross_validated_oof(frame, feature_cols, target):
    X = frame[feature_cols].copy()
    y = frame[target].astype(int).to_numpy()

    skf = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED)
    oof = np.full(len(frame), np.nan, dtype=float)
    fold_rows = []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
        pipe = build_pipeline()
        pipe.fit(X.iloc[tr_idx], y[tr_idx])
        prob = pipe.predict_proba(X.iloc[te_idx])[:, 1]
        oof[te_idx] = prob
        fold_rows.append(
            {
                "fold": fold,
                "auroc": roc_auc_score(y[te_idx], prob),
                "ap": average_precision_score(y[te_idx], prob),
            }
        )

    fold_df = pd.DataFrame(fold_rows)
    return {
        "oof": oof,
        "fold_df": fold_df,
        "mean_auroc": float(fold_df["auroc"].mean()),
        "std_auroc": float(fold_df["auroc"].std(ddof=1)),
        "mean_ap": float(fold_df["ap"].mean()),
        "std_ap": float(fold_df["ap"].std(ddof=1)),
    }


feature_sets = {
    "output_only": observable_output_cols,
    "early_trace_only": early_trace_features,
    "output_plus_early_trace": observable_output_cols + early_trace_features,
    "output_plus_full_trace": observable_output_cols + full_trace_features,
}

results = []
oof_scores = {}

for target in ["is_top1_error", "is_hard_miss"]:
    for name, cols in feature_sets.items():
        out = cross_validated_oof(analysis_df, cols, target)
        oof_scores[(target, name)] = out["oof"]
        results.append(
            {
                "target": target,
                "model": name,
                "n_features": len(cols),
                "auroc_mean": out["mean_auroc"],
                "auroc_std": out["std_auroc"],
                "ap_mean": out["mean_ap"],
                "ap_std": out["std_ap"],
            }
        )

model_results_df = pd.DataFrame(results).sort_values(["target", "auroc_mean"], ascending=[True, False])
display(model_results_df.round(4))

In [ ]:
def selective_curve_from_error_prob(error_prob, is_error, n_points=30):
    error_prob = np.asarray(error_prob, dtype=float)
    is_error = np.asarray(is_error, dtype=bool)
    order = np.argsort(error_prob)
    err_sorted = is_error[order]
    rows = []
    for cov in np.linspace(0.05, 1.0, n_points):
        keep = max(1, int(round(cov * len(err_sorted))))
        kept = err_sorted[:keep]
        rows.append(
            {
                "coverage": cov,
                "accuracy": float((~kept).mean()),
                "error_rate": float(kept.mean()),
            }
        )
    return pd.DataFrame(rows)


y_error = analysis_df["is_top1_error"].to_numpy()

curve_top1_prob = selective_curve_from_error_prob(1.0 - analysis_df["top1_prob"].to_numpy(), y_error)
curve_output_only = selective_curve_from_error_prob(oof_scores[("is_top1_error", "output_only")], y_error)
curve_output_full = selective_curve_from_error_prob(oof_scores[("is_top1_error", "output_plus_full_trace")], y_error)

plt.figure(figsize=(8, 5))
plt.plot(curve_top1_prob["coverage"], curve_top1_prob["accuracy"], label="1 - top1_prob")
plt.plot(curve_output_only["coverage"], curve_output_only["accuracy"], label="output_only model")
plt.plot(curve_output_full["coverage"], curve_output_full["accuracy"], label="output_plus_full_trace model")
plt.xlabel("Coverage kept")
plt.ylabel("Accuracy among kept examples")
plt.title("Selective prediction with learned reliability scores")
plt.legend()
plt.show()